# 22 · Silver: disaster events
**Owner: M** · Layer: silver · Runs after: 10, 20

**What:** turn `bronze.disaster_events` into two tables:
- `silver.disaster_events`: one typed row per event.
- `silver.disaster_event_states`: one row per event per state, because `regions` is a `;`-separated list.

| Rule | Problem (profiling evidence) | Treatment |
|---|---|---|
| DQ5 | Two date formats: `6/13/1989 0:00` and `24/10/1829 00:00:00` | Choose the format by pattern; record `date_format_used` |
| DQ18 | 3 shipwrecks dated 1 Jan 1753, SQL Server's minimum date | Treat as unknown date (NULL); flag `placeholder_date` |
| DQ17 | Cyclone titles break the `Category - place` pattern | Any prefix starting "Cyclone" is Cyclone |
| DQ11 | HTML entities such as `&deg;` in text | Unescape |
| DQ12 | Impact columns mostly blank | Blank = NULL (unknown), never 0 |

In [0]:
%run ./00_common

In [0]:
b = read("bronze", "disaster_events")
rows_in = b.count()

def de_date(col):
    raw = F.trim(F.col(col))
    has_seconds = raw.rlike(r"\d{2}:\d{2}:\d{2}$")               # day-first rows carry seconds; month-first rows don't
    ts = F.when(has_seconds, F.expr(f"try_to_timestamp(trim(`{col}`), 'dd/MM/yyyy HH:mm:ss')")) \
          .otherwise(F.expr(f"try_to_timestamp(trim(`{col}`), 'M/d/yyyy H:mm')"))
    return ts, F.when(has_seconds, "dd/MM/yyyy HH:mm:ss").otherwise("M/d/yyyy H:mm")

start_ts, start_fmt = de_date("startdate")
end_ts, _ = de_date("enddate")
prefix = F.trim(F.split("title", " - ").getItem(0))

def int_or_null(c):
    return F.expr(f"try_cast(regexp_replace(trim(`{c}`), ',', '') AS BIGINT)")

s = b.select(
    F.col("_row_num").alias("source_row"),
    F.col("id").alias("de_id"),
    blank_to_null("title").alias("title"),
    F.trim(F.when(F.size(F.split("title", " - ")) > 1, F.split("title", " - ").getItem(1)).otherwise(F.col("title"))).alias("title_place"),
    prefix.alias("title_prefix"),
    F.when(prefix.startswith("Cyclone"), "Cyclone").otherwise(prefix).alias("category"),            # DQ17
    unescape_html(F.col("description")).alias("description"),                                          # DQ11
    F.to_date(start_ts).alias("start_parsed"), F.to_date(end_ts).alias("end_parsed"), start_fmt.alias("date_format_used"),
    blank_to_null("regions").alias("regions_raw"),
    F.col("lat").cast("double").alias("lat"), F.col("lon").cast("double").alias("lon"),
    *[int_or_null(c).alias(a) for c, a in [("deaths", "deaths"), ("injuries", "injuries"), ("evacuated", "evacuated"),
                                           ("homeless", "homeless"), ("home_s_destroyed", "homes_destroyed"),
                                           ("home_s_damaged", "homes_damaged")]],
    num("insured_cost", False).alias("insured_cost_aud"),
)
s = (s.withColumn("placeholder_date", F.col("start_parsed") == F.lit(DATE_SENTINEL).cast("date"))       # DQ18
      .withColumn("start_date", F.when(F.col("placeholder_date"), None).otherwise(F.col("start_parsed")))
      .withColumn("end_date", F.when(F.col("placeholder_date"), None).otherwise(F.col("end_parsed")))
      .drop("start_parsed", "end_parsed"))

hz = read("silver", "xw_hazard").where("source = 'disaster_events'").select(F.col("source_value").alias("category"), "hazard_group", "hazard_family")
s = s.join(hz, "category", "left").withColumn("natural_hazard", F.col("hazard_group").isNotNull())

n_out = write(s, "silver", "disaster_events")
log_step("silver", "clean_disaster_events", "silver.disaster_events", rows_in, n_out, 0)

### One row per event per state
`Victoria;New South Wales;South Australia` becomes three rows, so an ICA event in any of those states can match.

In [0]:
xw = read("silver", "xw_state")
es = (read("silver", "disaster_events")
      .select("de_id", F.posexplode(F.split("regions_raw", ";")).alias("region_pos", "region"))
      .withColumn("source_value", F.upper(F.trim("region")))
      .join(xw, "source_value", "left")
      .select("de_id", "region_pos", F.trim("region").alias("region"), "state_code"))
n_states = write(es, "silver", "disaster_event_states")
log_step("silver", "explode_regions", "silver.disaster_event_states", n_states, n_states, 0,
         f"{es.where('state_code IS NULL').count()} region values not a state (e.g. Outside Australia)")

In [0]:
d = read("silver", "disaster_events")
display(d.groupBy("date_format_used").count())
display(d.agg(F.sum(F.col("placeholder_date").cast("int")).alias("placeholder_dates_DQ18"),
              F.sum(F.when(F.col("title_prefix") != F.col("category"), 1).otherwise(0)).alias("titles_normalised_DQ17"),
              F.sum(F.col("natural_hazard").cast("int")).alias("natural_hazard_events"),
              F.min("start_date").alias("earliest"), F.max("start_date").alias("latest")))